# H2 training: three-band versus six-band multispectral inputs

Trains matched three-band and six-band Detectree2 models under the leave-one-site-out design. Both arms use the multispectral pathway and differ only in the number of input bands.

## Setup

Installs the fixed Detectron2 and Detectree2 revisions used in the analysis. Restart the Colab runtime after installation.

In [ ]:
!pip -q install \
    "git+https://github.com/facebookresearch/detectron2.git@a2f4a8771ab77e8411c26b27f24f9489a28a2453"

!pip -q install \
    "git+https://github.com/PatBall1/detectree2.git@d9fb07f0dfb493f34def563c1ff896fecd59210d"

!pip -q install rasterio geopandas pyyaml

## Paths and configuration

Defines the source-data, preprocessing and model-output paths together with the shared H2 training parameters.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import inspect
import json
import os
import shutil
import traceback
import urllib.request

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import torch
import yaml

from rasterio.mask import mask as rio_mask
from shapely.ops import unary_union

from detectron2.data import (
    DatasetCatalog,
    MetadataCatalog,
    build_detection_test_loader,
    build_detection_train_loader,
)
import detectron2.data.transforms as T

from detectree2.models.train import (
    FlexibleDatasetMapper,
    MyTrainer,
    get_tree_dicts,
    setup_cfg,
)
from detectree2.preprocessing.tiling import tile_data


DATA_ROOT = Path(
    "/content/drive/MyDrive/Congo basin/congo"
)
H2_RUNS = Path(
    "/content/drive/MyDrive/Congo basin/H2_runs"
)

WORK = Path("/content/h2_training_work")
STACKS = WORK / "stacks"
CROPS = WORK / "crops"
TILES = WORK / "tiles"
KEEP = WORK / "keep"
LOCAL_RUNS = WORK / "runs"

REBUILD_INPUTS = False
OVERWRITE_MODELS = False
CLEAN_LOCAL_RUNS = True

if REBUILD_INPUTS and WORK.exists():
    shutil.rmtree(WORK)

for directory in (
    STACKS,
    CROPS,
    TILES,
    KEEP,
    LOCAL_RUNS,
    H2_RUNS,
):
    directory.mkdir(parents=True, exist_ok=True)


SITES = ["lokoue", "dzanga", "mbeli"]
STRATA = ["tall", "mid", "small"]

TILE_WIDTH = 50
BUFFER = 25
MIN_CHECKPOINT_MB = 400

BASE_MODEL = (
    "COCO-InstanceSegmentation/"
    "mask_rcnn_R_101_FPN_3x.yaml"
)

PRETRAINED_MODEL = Path(
    "/content/230103_randresize_full.pth"
)

ARMS = {
    "three_band": {
        "label": "Three-band MS",
        "imgmode": "ms",
        "num_bands": 3,
        # Retained for compatibility with existing outputs.
        "run_prefix": "rgb",
    },
    "six_band": {
        "label": "Six-band MS",
        "imgmode": "ms",
        "num_bands": 6,
        "run_prefix": "ms",
    },
}

BAND_NAMES = [
    "red",
    "green",
    "blue",
    "near-infrared",
    "red-edge",
    "deep-blue",
]

paths = {
    site: {
        "rgb": (
            DATA_ROOT
            / site
            / f"{site}_RGB.TIF"
        ),
        "ned": (
            DATA_ROOT
            / site
            / f"{site}_NED.TIF"
        ),
        "stack": (
            STACKS
            / f"{site}_six_band_uint16.tif"
        ),
        "mask": (
            DATA_ROOT
            / site
            / f"mask_{site}.gpkg"
        ),
        "aoi": {
            stratum: (
                DATA_ROOT
                / site
                / f"{site}_aoi_{stratum}.gpkg"
            )
            for stratum in STRATA
        },
        "crowns": {
            stratum: (
                DATA_ROOT
                / site
                / f"crowns_{site}_{stratum}.gpkg"
            )
            for stratum in STRATA
        },
    }
    for site in SITES
}

## Input validation

Checks the source rasters, spatial annotations and pretrained checkpoint before preprocessing.

In [ ]:
required_files = []

for site in SITES:
    required_files.extend([
        paths[site]["rgb"],
        paths[site]["ned"],
        paths[site]["mask"],
        *paths[site]["aoi"].values(),
        *paths[site]["crowns"].values(),
    ])

missing_files = [
    path
    for path in required_files
    if not path.exists()
]

assert not missing_files, (
    "Missing required files:\n"
    + "\n".join(map(str, missing_files))
)

for site in SITES:
    with (
        rasterio.open(paths[site]["rgb"]) as rgb,
        rasterio.open(paths[site]["ned"]) as ned,
    ):
        assert rgb.count >= 3 and ned.count >= 3
        assert rgb.crs == ned.crs, f"{site}: CRS mismatch"
        assert rgb.shape == ned.shape, f"{site}: shape mismatch"
        assert rgb.transform == ned.transform, (
            f"{site}: transform mismatch"
        )
        assert rgb.bounds == ned.bounds, (
            f"{site}: bounds mismatch"
        )
        assert rgb.res == ned.res, (
            f"{site}: resolution mismatch"
        )
        assert rgb.dtypes[:3] == ned.dtypes[:3]
        assert set(rgb.dtypes[:3]) == {"uint16"}

        print(
            f"{site}: {rgb.width} × {rgb.height} pixels, "
            f"{rgb.res[0]:.3f} m resolution, "
            f"{rgb.crs}"
        )

if not PRETRAINED_MODEL.exists():
    urllib.request.urlretrieve(
        "https://zenodo.org/records/10522461/"
        "files/230103_randresize_full.pth",
        PRETRAINED_MODEL,
    )

assert PRETRAINED_MODEL.stat().st_size / 1e6 > 400, (
    "Pretrained checkpoint is incomplete"
)

## Six-band inputs

Combines the native 16-bit products in the order red, green, blue, near-infrared, red-edge and deep-blue. No additional stretch is applied before multispectral tiling.

In [ ]:
def build_six_band_stack(site):
    output_path = paths[site]["stack"]

    if output_path.exists() and not REBUILD_INPUTS:
        with rasterio.open(output_path) as raster:
            valid_output = (
                raster.count == 6
                and raster.dtypes == ("uint16",) * 6
                and raster.nodata == 0
            )

        if valid_output:
            print(
                f"{site}: retaining existing "
                "six-band stack"
            )
            return output_path

        output_path.unlink()

    with (
        rasterio.open(paths[site]["rgb"]) as rgb,
        rasterio.open(paths[site]["ned"]) as ned,
    ):
        metadata = rgb.meta.copy()
        metadata.update(
            driver="GTiff",
            count=6,
            dtype="uint16",
            nodata=0,
            tiled=True,
            blockxsize=512,
            blockysize=512,
            compress="deflate",
        )

        with rasterio.open(
            output_path,
            "w",
            **metadata,
        ) as destination:
            for _, window in rgb.block_windows(1):
                values = np.concatenate([
                    rgb.read(
                        [1, 2, 3],
                        window=window,
                    ),
                    ned.read(
                        [1, 2, 3],
                        window=window,
                    ),
                ], axis=0)

                destination.write(
                    values,
                    window=window,
                )

    with rasterio.open(output_path) as raster:
        assert raster.count == 6
        assert raster.dtypes == ("uint16",) * 6
        assert raster.nodata == 0

    print(f"{site}: six-band stack created")
    return output_path


for site in SITES:
    build_six_band_stack(site)

## Valid regions

Reconstructs the AOI-minus-mask regions used for H1 and retains the same crown annotations for both H2 arms.

In [ ]:
prepped = {}

for site in SITES:
    with rasterio.open(paths[site]["stack"]) as raster:
        raster_crs = raster.crs

    boundary_mask = gpd.read_file(
        paths[site]["mask"]
    ).to_crs(raster_crs)
    boundary_mask.geometry = (
        boundary_mask.geometry.buffer(0)
    )

    prepped[site] = {}

    for stratum in STRATA:
        aoi = gpd.read_file(
            paths[site]["aoi"][stratum]
        ).to_crs(raster_crs)

        crowns = gpd.read_file(
            paths[site]["crowns"][stratum]
        ).to_crs(raster_crs)

        aoi.geometry = aoi.geometry.buffer(0)
        crowns.geometry = crowns.geometry.buffer(0)

        assert aoi.geometry.is_valid.all()
        assert crowns.geometry.is_valid.all()

        aoi_union = unary_union(aoi.geometry)

        inside_aoi = (
            crowns.geometry.centroid.within(
                aoi_union
            )
        )

        assert inside_aoi.all(), (
            f"{site}/{stratum}: crown centroid "
            "outside its AOI"
        )

        keep = gpd.overlay(
            aoi,
            boundary_mask,
            how="difference",
        )
        keep.geometry = keep.geometry.buffer(0)

        assert len(keep) > 0
        assert keep.geometry.is_valid.all()

        keep_path = (
            KEEP
            / f"keep_{site}_{stratum}.gpkg"
        )
        keep.to_file(
            keep_path,
            driver="GPKG",
        )

        prepped[site][stratum] = {
            "crowns": crowns,
            "keep_path": keep_path,
        }

        print(
            f"{site}/{stratum}: "
            f"{len(crowns)} crowns"
        )

## Six-band tiling

Creates the shared multispectral tiles using the same spatial geometry as H1. Detectree2 applies its per-band 1st–99th percentile scaling during tiling.

In [ ]:
def tile_six_band_aoi(site, stratum):
    crowns = prepped[site][stratum]["crowns"]
    keep_path = prepped[site][stratum]["keep_path"]

    aoi = gpd.read_file(
        paths[site]["aoi"][stratum]
    ).to_crs(crowns.crs)

    crop_path = (
        CROPS
        / f"{site}_{stratum}_six_band.tif"
    )

    if not crop_path.exists():
        with rasterio.open(
            paths[site]["stack"]
        ) as source:
            image, transform = rio_mask(
                source,
                aoi.geometry.buffer(
                    2,
                    join_style=2,
                ),
                crop=True,
                nodata=0,
            )

            metadata = source.meta.copy()
            metadata.update(
                height=image.shape[1],
                width=image.shape[2],
                transform=transform,
                nodata=0,
            )

        with rasterio.open(
            crop_path,
            "w",
            **metadata,
        ) as destination:
            destination.write(image)

    output_directory = (
        TILES
        / f"{site}_{stratum}_six_band_"
          f"{TILE_WIDTH}_{BUFFER}"
    )

    if not output_directory.exists():
        tile_data(
            img_path=str(crop_path),
            out_dir=str(output_directory),
            buffer=BUFFER,
            tile_width=TILE_WIDTH,
            tile_height=TILE_WIDTH,
            crowns=crowns,
            threshold=0.0,
            nan_threshold=1.0,
            full_coverage=False,
            mode="ms",
            mask_path=str(keep_path),
            use_convex_mask=False,
            enhance_rgb_contrast=False,
            tile_placement="grid",
            multithreaded=True,
            ignore_bands_indices=[],
        )

    tif_count = len(
        list(output_directory.glob("*.tif"))
    )
    annotation_count = len(
        list(output_directory.glob("*.geojson"))
    )

    assert tif_count == 25, (
        f"{site}/{stratum}: expected 25 rasters, "
        f"found {tif_count}"
    )
    assert annotation_count == 25, (
        f"{site}/{stratum}: expected 25 annotations, "
        f"found {annotation_count}"
    )

    print(
        f"{site}/{stratum}: "
        f"{tif_count} six-band tiles"
    )
    return output_directory


six_band_directories = {
    (site, stratum): tile_six_band_aoi(
        site,
        stratum,
    )
    for site in SITES
    for stratum in STRATA
}

six_band_total = sum(
    len(list(directory.glob("*.geojson")))
    for directory in six_band_directories.values()
)

assert six_band_total == 225

## Three-band multispectral inputs

Creates three-band GeoTIFF copies from bands 1–3 of each six-band tile. Annotation paths remain linked to GeoTIFFs so the inputs are read through Detectree2’s multispectral pathway.

In [ ]:
def create_three_band_ms_clone(
    source_directory,
    output_directory,
):
    if output_directory.exists():
        raster_count = len(
            list(output_directory.glob("*.tif"))
        )
        annotation_count = len(
            list(output_directory.glob("*.geojson"))
        )

        assert raster_count == 25, (
            f"{output_directory}: expected 25 rasters, "
            f"found {raster_count}"
        )
        assert annotation_count == 25, (
            f"{output_directory}: expected 25 annotations, "
            f"found {annotation_count}"
        )

        return output_directory

    output_directory.mkdir(
        parents=True,
        exist_ok=False,
    )

    source_rasters = sorted(
        source_directory.glob("*.tif")
    )
    source_annotations = sorted(
        source_directory.glob("*.geojson")
    )

    assert len(source_rasters) == 25
    assert len(source_annotations) == 25

    raster_by_stem = {
        raster.stem: raster
        for raster in source_rasters
    }

    for stem, source_raster in raster_by_stem.items():
        output_raster = (
            output_directory
            / source_raster.name
        )

        with rasterio.open(source_raster) as source:
            data = source.read([1, 2, 3])
            metadata = source.meta.copy()
            metadata.update(count=3)

        with rasterio.open(
            output_raster,
            "w",
            **metadata,
        ) as destination:
            destination.write(data)

        source_annotation = (
            source_directory
            / f"{stem}.geojson"
        )

        assert source_annotation.exists(), (
            f"Missing annotation for {source_raster.name}"
        )

        with open(
            source_annotation,
            "r",
        ) as annotation_file:
            annotation = json.load(annotation_file)

        assert "imagePath" in annotation, (
            f"{source_annotation.name}: "
            "missing imagePath"
        )

        annotation["imagePath"] = str(
            output_raster.resolve()
        )

        output_annotation = (
            output_directory
            / source_annotation.name
        )

        with open(
            output_annotation,
            "w",
        ) as annotation_file:
            json.dump(
                annotation,
                annotation_file,
                indent=2,
            )

    assert (
        len(list(output_directory.glob("*.tif")))
        == 25
    )
    assert (
        len(list(output_directory.glob("*.geojson")))
        == 25
    )

    return output_directory


three_band_directories = {}

for site in SITES:
    for stratum in STRATA:
        source_directory = six_band_directories[
            (site, stratum)
        ]

        output_directory = (
            TILES
            / f"{site}_{stratum}_three_band_ms_"
              f"{TILE_WIDTH}_{BUFFER}"
        )

        three_band_directories[
            (site, stratum)
        ] = create_three_band_ms_clone(
            source_directory,
            output_directory,
        )

        print(
            f"{site}/{stratum}: "
            "three-band MS clone ready"
        )

three_band_total = sum(
    len(list(directory.glob("*.geojson")))
    for directory in three_band_directories.values()
)

assert three_band_total == 225

## Matched-input verification

Checks every three-band tile against bands 1–3 of its six-band counterpart, including pixel values and spatial metadata.

In [ ]:
verified_pairs = 0

for site in SITES:
    for stratum in STRATA:
        three_directory = (
            three_band_directories[
                (site, stratum)
            ]
        )

        six_directory = (
            six_band_directories[
                (site, stratum)
            ]
        )

        six_rasters = {
            path.stem: path
            for path
            in six_directory.glob(
                "*.tif"
            )
        }

        for three_path in sorted(
            three_directory.glob(
                "*.tif"
            )
        ):
            assert (
                three_path.stem
                in six_rasters
            )

            six_path = six_rasters[
                three_path.stem
            ]

            with (
                rasterio.open(
                    three_path
                ) as three,
                rasterio.open(
                    six_path
                ) as six,
            ):
                assert three.count == 3
                assert six.count == 6

                assert (
                    three.width
                    == six.width
                )
                assert (
                    three.height
                    == six.height
                )
                assert (
                    three.transform
                    == six.transform
                )
                assert (
                    three.crs
                    == six.crs
                )
                assert (
                    three.nodata
                    == six.nodata
                )
                assert (
                    three.dtypes
                    == six.dtypes[:3]
                )

                assert np.array_equal(
                    three.read(),
                    six.read(
                        [1, 2, 3]
                    ),
                ), (
                    "Band mismatch: "
                    f"{three_path.name}"
                )

            annotation_path = (
                three_directory
                / (
                    f"{three_path.stem}"
                    ".geojson"
                )
            )

            with open(
                annotation_path,
                "r",
            ) as annotation_file:
                annotation = json.load(
                    annotation_file
                )

            linked_image = Path(
                annotation["imagePath"]
            )

            assert (
                linked_image.resolve()
                == three_path.resolve()
            )

            assert (
                linked_image
                .suffix
                .lower()
                == ".tif"
            )

            verified_pairs += 1

assert verified_pairs == 225

print(
    "PASS: 225 matched three-band "
    "and six-band tile pairs verified"
)

## Leave-one-site-out partitions

Registers matched three-band and six-band datasets. Each fold contains 100 training, 50 validation and 75 held-out test tiles.

In [ ]:
arm_directories = {
    "three_band": three_band_directories,
    "six_band": six_band_directories,
}


def register_fold(
    directories,
    holdout,
    tag,
    validation_stratum="mid",
):
    dataset_names = {
        split: f"{tag}_{split}"
        for split in ("train", "validation", "test")
    }

    for name in dataset_names.values():
        if name in DatasetCatalog.list():
            DatasetCatalog.remove(name)

        if name in MetadataCatalog.list():
            MetadataCatalog.remove(name)

    partitions = {
        "train": [],
        "validation": [],
        "test": [],
    }

    for (site, stratum), directory in directories.items():
        records = get_tree_dicts(str(directory))

        if site == holdout:
            partitions["test"].extend(records)
        elif stratum == validation_stratum:
            partitions["validation"].extend(records)
        else:
            partitions["train"].extend(records)

    counts = {
        split: len(records)
        for split, records in partitions.items()
    }

    assert counts == {
        "train": 100,
        "validation": 50,
        "test": 75,
    }, (
        f"{tag}: unexpected fold composition "
        f"{counts}"
    )

    for split, records in partitions.items():
        name = dataset_names[split]

        DatasetCatalog.register(
            name,
            lambda records=records: records,
        )

        MetadataCatalog.get(name).set(
            thing_classes=["tree"]
        )

    print(
        f"{tag}: train {counts['train']} | "
        f"validation {counts['validation']} | "
        f"test {counts['test']}"
    )

    return dataset_names


registered_folds = {}

for arm in ARMS:
    for holdout in SITES:
        tag = f"h2_{arm}_{holdout}"

        registered_folds[
            (arm, holdout)
        ] = register_fold(
            arm_directories[arm],
            holdout,
            tag,
        )

## Mapper verification

Confirms that both arms use the multispectral mapper and produce finite three-channel and six-channel tensors.

In [ ]:
required_parameters = {
    "imgmode",
    "num_bands",
    "resize",
    "eval_period",
}

assert required_parameters.issubset(
    inspect.signature(setup_cfg).parameters
), "Unexpected Detectree2 API"


for arm in ARMS:
    settings = ARMS[arm]
    dataset_names = registered_folds[
        (arm, "lokoue")
    ]

    configuration = setup_cfg(
        base_model=BASE_MODEL,
        trains=(dataset_names["train"],),
        tests=(dataset_names["validation"],),
        update_model=None,
        workers=2,
        ims_per_batch=2,
        max_iter=1,
        eval_period=0,
        resize="rand_fixed",
        imgmode="ms",
        num_bands=settings["num_bands"],
        out_dir=(
            f"/content/h2_mapper_check_{arm}"
        ),
    )

    loader = build_detection_train_loader(
        configuration,
        mapper=FlexibleDatasetMapper(
            configuration,
            is_train=True,
        ),
    )

    sample = next(iter(loader))[0]["image"]

    assert sample.shape[0] == settings["num_bands"]
    assert torch.isfinite(sample.float()).all()

    print(
        f"{settings['label']}: "
        f"shape {tuple(sample.shape)}, "
        f"dtype {sample.dtype}"
    )

## Model configuration

Uses the same model architecture, optimisation settings, validation resizing and early-stopping procedure as H1.

In [ ]:
def fixed_validation_loader(
    cls,
    configuration,
    dataset_name,
):
    mapper = FlexibleDatasetMapper(
        configuration,
        is_train=False,
        augmentations=[
            T.ResizeShortestEdge(
                [1000, 1000],
                1333,
            )
        ],
    )

    return build_detection_test_loader(
        configuration,
        dataset_name,
        mapper=mapper,
    )


MyTrainer.build_test_loader = classmethod(
    fixed_validation_loader
)


def checkpoint_input_channels(checkpoint_path):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
    )

    state_dict = checkpoint.get(
        "model",
        checkpoint,
    )

    convolution_keys = [
        key
        for key in state_dict
        if key.endswith(
            "backbone.bottom_up.stem.conv1.weight"
        )
        or key.endswith("stem.conv1.weight")
    ]

    assert len(convolution_keys) == 1, (
        f"{checkpoint_path}: unexpected conv1 keys "
        f"{convolution_keys}"
    )

    weights = state_dict[convolution_keys[0]]
    return int(weights.shape[1])


assert checkpoint_input_channels(
    PRETRAINED_MODEL
) == 3

## Initialisation verification

Checks the input-layer dimensions after loading the pretrained checkpoint. The six-band model retains pretrained RGB weights in its first three channels and random initialisation in the additional channels.

In [ ]:
pretrained_checkpoint = torch.load(
    PRETRAINED_MODEL,
    map_location="cpu",
)

pretrained_state = pretrained_checkpoint["model"]

pretrained_conv1_key = next(
    key
    for key in pretrained_state
    if key.endswith("stem.conv1.weight")
)

pretrained_conv1 = pretrained_state[
    pretrained_conv1_key
].detach().cpu()


def verify_initialisation(trainer, arm):
    settings = ARMS[arm]

    weights = (
        trainer.model
        .backbone
        .bottom_up
        .stem
        .conv1
        .weight
        .detach()
        .cpu()
    )

    expected_shape = (
        64,
        settings["num_bands"],
        7,
        7,
    )

    assert tuple(weights.shape) == expected_shape
    assert torch.isfinite(weights).all()

    assert torch.allclose(
        weights[:, :3],
        pretrained_conv1.to(weights.dtype),
        atol=1e-6,
    ), (
        f"{arm}: RGB channels do not match "
        "the pretrained checkpoint"
    )

    if arm == "six_band":
        additional_weights = weights[:, 3:6]

        assert torch.isfinite(
            additional_weights
        ).all()
        assert additional_weights.std() > 0

        for additional_channel in range(3):
            for pretrained_channel in range(3):
                assert not torch.equal(
                    additional_weights[
                        :,
                        additional_channel,
                    ],
                    pretrained_conv1[
                        :,
                        pretrained_channel,
                    ].to(weights.dtype),
                ), (
                    "Additional channel duplicates "
                    "a pretrained RGB channel"
                )

    print(
        f"{settings['label']}: "
        f"conv1 {tuple(weights.shape)} verified"
    )

## Checkpoint selection

Selects the checkpoint corresponding to the highest validation segmentation AP50 and verifies the saved model before persistence.

In [ ]:
def best_ap50_iteration(metrics_path):
    metrics_path = Path(metrics_path)

    assert metrics_path.exists(), (
        f"Missing metrics file: {metrics_path}"
    )

    evaluations = []

    with open(metrics_path, "r") as metrics_file:
        for line in metrics_file:
            record = json.loads(line)

            if "segm/AP50" in record:
                evaluations.append((
                    float(record["segm/AP50"]),
                    int(record["iteration"]),
                ))

    assert evaluations, (
        f"No segmentation AP50 values in "
        f"{metrics_path}"
    )

    return max(
        evaluations,
        key=lambda result: result[0],
    )


def promote_best_checkpoint(output_directory):
    output_directory = Path(output_directory)

    best_ap50, best_iteration = (
        best_ap50_iteration(
            output_directory / "metrics.json"
        )
    )

    candidates = []

    for checkpoint in output_directory.glob(
        "model_*.pth"
    ):
        iteration_text = checkpoint.stem.replace(
            "model_",
            "",
        )

        if iteration_text.isdigit():
            checkpoint_iteration = int(
                iteration_text
            )

            candidates.append((
                abs(
                    checkpoint_iteration
                    - best_iteration
                ),
                checkpoint_iteration,
                checkpoint,
            ))

    assert candidates, (
        f"{output_directory.name}: "
        "no iteration checkpoints found"
    )

    (
        offset,
        checkpoint_iteration,
        source_checkpoint,
    ) = min(candidates)

    assert offset <= 1, (
        f"{output_directory.name}: nearest "
        f"checkpoint iteration "
        f"{checkpoint_iteration} differs from "
        f"best evaluation iteration "
        f"{best_iteration}"
    )

    best_checkpoint = (
        output_directory
        / "model_best.pth"
    )

    shutil.copy2(
        source_checkpoint,
        best_checkpoint,
    )

    checkpoint_size_mb = (
        best_checkpoint.stat().st_size
        / 1e6
    )

    assert checkpoint_size_mb >= MIN_CHECKPOINT_MB

    return {
        "best_ap50": best_ap50,
        "best_iteration": best_iteration,
        "checkpoint_iteration": checkpoint_iteration,
        "checkpoint_source": source_checkpoint.name,
        "checkpoint_size_mb": checkpoint_size_mb,
    }

## Existing-model validation

Existing checkpoints and metrics are checked for completeness and the expected input-band count. Archived configurations are also validated when available; legacy runs without config.yaml are retained with the missing metadata reported explicitly.

In [ ]:
def run_directory(
    arm,
    holdout,
):
    prefix = ARMS[arm][
        "run_prefix"
    ]

    return (
        H2_RUNS
        / f"{prefix}_{holdout}"
    )


def read_saved_configuration(
    directory,
):
    configuration_path = (
        directory
        / "config.yaml"
    )

    if not configuration_path.exists():
        return None

    with open(
        configuration_path,
        "r",
    ) as configuration_file:
        configuration = yaml.safe_load(
            configuration_file
        )

    assert isinstance(
        configuration,
        dict,
    ), (
        f"{configuration_path}: "
        "configuration is empty or invalid"
    )

    return configuration


def validate_saved_run(
    arm,
    holdout,
):
    directory = run_directory(
        arm,
        holdout,
    )

    checkpoint = (
        directory
        / "model_best.pth"
    )

    metrics = (
        directory
        / "metrics.json"
    )

    assert checkpoint.exists(), (
        f"{directory}: "
        "missing model_best.pth"
    )

    assert metrics.exists(), (
        f"{directory}: "
        "missing metrics.json"
    )

    checkpoint_size_mb = (
        checkpoint.stat().st_size
        / 1e6
    )

    assert (
        checkpoint_size_mb
        >= MIN_CHECKPOINT_MB
    ), (
        f"{checkpoint}: "
        "incomplete checkpoint"
    )

    expected_bands = ARMS[
        arm
    ]["num_bands"]

    checkpoint_bands = (
        checkpoint_input_channels(
            checkpoint
        )
    )

    assert (
        checkpoint_bands
        == expected_bands
    ), (
        f"{directory}: checkpoint has "
        f"{checkpoint_bands} input "
        f"channels, expected "
        f"{expected_bands}"
    )

    configuration = (
        read_saved_configuration(
            directory
        )
    )

    if configuration is None:
        configuration_status = (
            "not archived"
        )

        saved_mode = None
        saved_bands = None

        print(
            f"NOTE: {directory.name}: "
            "legacy config.yaml was "
            "not archived"
        )

    else:
        saved_mode = str(
            configuration.get(
                "IMGMODE",
                "",
            )
        ).lower()

        saved_bands = int(
            configuration.get(
                "INPUT",
                {},
            ).get(
                "NUM_IN_CHANNELS",
                -1,
            )
        )

        assert saved_mode == "ms", (
            f"{directory}: saved "
            f"IMGMODE is "
            f"{saved_mode!r}, "
            "expected 'ms'"
        )

        assert (
            saved_bands
            == expected_bands
        ), (
            f"{directory}: saved "
            f"NUM_BANDS is "
            f"{saved_bands}, expected "
            f"{expected_bands}"
        )

        configuration_status = (
            "verified"
        )

    best_ap50, best_iteration = (
        best_ap50_iteration(
            metrics
        )
    )

    return {
        "directory": str(
            directory
        ),
        "checkpoint": str(
            checkpoint
        ),
        "best_ap50": best_ap50,
        "best_iteration": (
            best_iteration
        ),
        "checkpoint_size_mb": (
            checkpoint_size_mb
        ),
        "checkpoint_bands": (
            checkpoint_bands
        ),
        "configuration_status": (
            configuration_status
        ),
        "saved_imgmode": (
            saved_mode
        ),
        "saved_num_bands": (
            saved_bands
        ),
    }

## Model training

Trains one three-band and one six-band model for each held-out site. Completed compatible models are retained unless overwriting is enabled.

In [ ]:
def train_fold(arm, holdout):
    settings = ARMS[arm]
    dataset_names = registered_folds[
        (arm, holdout)
    ]

    persistent_directory = run_directory(
        arm,
        holdout,
    )
    local_directory = (
        LOCAL_RUNS
        / f"{arm}_{holdout}"
    )

    persistent_checkpoint = (
        persistent_directory
        / "model_best.pth"
    )

    if (
        persistent_checkpoint.exists()
        and not OVERWRITE_MODELS
    ):
        existing = validate_saved_run(
            arm,
            holdout,
        )

        print(
            f"{settings['label']}/{holdout}: "
            "retaining existing model"
        )

        return {
            "arm": arm,
            "holdout": holdout,
            "status": "retained",
            **existing,
        }

    if (
        persistent_directory.exists()
        and any(persistent_directory.iterdir())
        and not OVERWRITE_MODELS
    ):
        raise RuntimeError(
            f"{persistent_directory} contains an "
            "incomplete or incompatible run. "
            "Inspect it before enabling "
            "OVERWRITE_MODELS."
        )

    if local_directory.exists():
        if OVERWRITE_MODELS:
            shutil.rmtree(local_directory)
        elif any(local_directory.iterdir()):
            raise RuntimeError(
                f"{local_directory} contains a "
                "partial local run. Inspect or "
                "remove it before retraining."
            )

    local_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    configuration = setup_cfg(
        base_model=BASE_MODEL,
        trains=(dataset_names["train"],),
        tests=(dataset_names["validation"],),
        update_model=str(PRETRAINED_MODEL),
        workers=2,
        ims_per_batch=2,
        base_lr=0.0003389,
        backbone_freeze=0,
        max_iter=3000,
        eval_period=100,
        resize="rand_fixed",
        imgmode="ms",
        num_bands=settings["num_bands"],
        out_dir=str(local_directory),
    )

    configuration.SOLVER.CHECKPOINT_PERIOD = 100

    with open(
        local_directory / "config.yaml",
        "w",
    ) as configuration_file:
        configuration_file.write(
            configuration.dump()
        )

    print(
        f"\nTraining {settings['label']} | "
        f"hold out {holdout}"
    )

    trainer = MyTrainer(
        configuration,
        patience=5,
    )
    trainer.resume_or_load(
        resume=False
    )

    verify_initialisation(
        trainer,
        arm,
    )

    trainer.train()

    selection = promote_best_checkpoint(
        local_directory
    )

    if persistent_directory.exists():
        shutil.rmtree(
            persistent_directory
        )

    persistent_directory.mkdir(
        parents=True,
        exist_ok=False,
    )

    for filename in (
        "model_best.pth",
        "metrics.json",
        "config.yaml",
    ):
        source = local_directory / filename

        assert source.exists(), (
            f"Missing training output: {source}"
        )

        shutil.copy2(
            source,
            persistent_directory / filename,
        )

    os.sync()

    saved = validate_saved_run(
        arm,
        holdout,
    )

    if (
        CLEAN_LOCAL_RUNS
        and local_directory.exists()
    ):
        shutil.rmtree(local_directory)

    print(
        f"{settings['label']}/{holdout}: "
        f"AP50={selection['best_ap50']:.2f}, "
        f"iteration={selection['best_iteration']}"
    )

    return {
        "arm": arm,
        "holdout": holdout,
        "status": "trained",
        **saved,
    }

## Run all H2 folds

Runs the three-band folds followed by the six-band folds, preserving completed models if a later fold fails.

In [ ]:
training_summary = []

try:
    for arm in ("three_band", "six_band"):
        for holdout in SITES:
            result = train_fold(
                arm,
                holdout,
            )
            training_summary.append(result)

except Exception as error:
    print(
        f"Training stopped: "
        f"{type(error).__name__}: {error}"
    )
    traceback.print_exc()

    raise

## Training summary

Verifies the six expected model outputs and exports their configurations, checkpoint dimensions and best validation AP50 values.

In [ ]:
summary_table = pd.DataFrame(
    training_summary
)

expected_runs = {
    (arm, holdout)
    for arm in ARMS
    for holdout in SITES
}

completed_runs = {
    (row["arm"], row["holdout"])
    for row in training_summary
}

assert completed_runs == expected_runs, (
    "Not all six H2 folds completed or were retained"
)

for arm, holdout in expected_runs:
    validate_saved_run(
        arm,
        holdout,
    )

summary_path = (
    H2_RUNS
    / "h2_training_summary.csv"
)

summary_table.to_csv(
    summary_path,
    index=False,
)

display(
    summary_table[
        [
            "arm",
            "holdout",
            "status",
            "best_ap50",
            "best_iteration",
            "checkpoint_size_mb",
        ]
    ].round(3)
)

print("PASS: six H2 models verified")
print("Training summary:", summary_path)